In [ ]:
import sys
sys.path.insert(0, '.')
import figio as fx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Every number here comes from scoring/*/; nothing is recomputed. Bars are the mean
# over three seeds and the whisker is their SD, each seed being a mean over its
# five folds. Re-run the scoring notebooks first if these files are stale.

from matplotlib.gridspec import GridSpec

# plot1: the full-model comparison on top, the three narrower datasets below.
# template_v2 carries every model and its display order.
#
# refs=True keeps NetMHCIIpan-4.3 and MixMHC2pred-2.0, which pipeline.ref_level()
# scores on each analysis's own test set. They carry roc_auc and pr_auc only: both
# emit a percentile rank rather than a calibrated probability, so the 0.5 cut the
# other three metrics use has no meaning for them - a %Rank of 0.5 is a strong
# binder, not a coin flip. `bars` skips a metric a method has no value for rather
# than drawing a zero-height bar, which would read as a score of zero.
#
# The two IC50 panels carry NetMHCIIpan's BA head (%Rank_BA), the other two its EL
# head (%Rank_EL): the heads have different training targets and the IC50 test set
# is measured affinity. 3_ic/config.py sets that; see
# docs/notes/2026-09-01-netmhciipan-el-vs-ba.md. The IC50 y-axis tops out at 0.95
# rather than 0.90 because the BA row reaches 0.909 and would otherwise be clipped.
# NetMHCIIpan-4.3 does have an affinity head, so it stays in (c) and (d) - but on
# its own training target, which is why its bar there is an upper bound and not a
# like-for-like result. Both facts belong in the caption.
#
# The top panel shows every model (plotted_only=False). The bottom row shows the
# eight marked plot=True in template_v2: the five representations MS and IC50 were
# originally run on - BLOSUM62, Chai-1, ESMC 300M, ESM3 Small, DeepNeo - plus
# ESM3 Large, whose MS and IC50 runs landed 2026-09-03 and which leads the
# Qualitative and MS columns, plus the two published tools. The remaining four
# (AlphaFold 3, Boltz-1, ESMC 600M, ESMC 6B, ESM3 Medium) came from the 260830
# fill-in; they stay in tab:perf-summary and the supplementary tables, so keeping
# them out here holds the three panels to the comparison the text makes. Both rows
# read the same template, so a method sits in the same position in every panel it
# appears in.
#
# MixMHC2pred-2.0 is dropped from the two IC50 panels. It is trained only on
# naturally presented peptides and has no affinity head at all
# (docs/notes/2026-09-01-mixmhc2pred-no-affinity-head.md), so a bar for it on a
# measured-IC50 target is not a comparison anyone would make. It stays in (a) and
# (b), whose targets it was built for, and its measured IC50 values stay in
# tab:perf-summary and Supplementary Tables S9/S10/S14 for completeness.
TMPL = 'template_v2.csv'
IC50_DROP = ['mixmhcpred']
df1 = fx.rep('1_whole', tmpl=TMPL, plotted_only=False, refs=True)
df2 = fx.rep('2_ms',    tmpl=TMPL, plotted_only=True , refs=True, dir='plots_ql')
df3 = fx.rep('3_ic',    tmpl=TMPL, plotted_only=True , refs=True, dir='plots_500')
df4 = fx.rep('3_ic',    tmpl=TMPL, plotted_only=True , refs=True, dir='plots_1000')
df3 = df3[~df3['model'].isin(IC50_DROP)].reset_index(drop=True)
df4 = df4[~df4['model'].isin(IC50_DROP)].reset_index(drop=True)

METRICS = ['roc_auc', 'pr_auc', 'f1', 'accuracy', 'mcc']
custom_palette = ['#D72000FF', '#EE6100FF', '#FFAD0AFF', '#1BB6AFFF', '#9093A2FF', '#132157FF']
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=custom_palette)

fig = plt.figure(figsize=(12, 8))
gs = GridSpec(2, 3, height_ratios=[1, 1], hspace=0.6, wspace=0.2, width_ratios=[8, 7, 7])

def bars(ax, df, title, ylim):
    for i, m in enumerate(METRICS):
        have = df[m].notna().values
        xs = np.array([x + i * 0.15 for x in range(len(df))])
        sd = df[f'{m}_sd'] if f'{m}_sd' in df else pd.Series(index=df.index, dtype=float)
        ax.bar(xs[have], df[m].values[have], width=0.15,
               yerr=sd.fillna(0).values[have], capsize=2,
               error_kw={'lw': 0.7, 'ecolor': '0.3'},
               color=custom_palette[i % len(custom_palette)])
    ax.set_xticks([x + 0.3 for x in range(len(df))])
    ax.set_xticklabels(df['full_name'], rotation=45, ha='right')
    ax.set_title(title)
    ax.set_ylim(*ylim)
    ax.grid(axis='y', linestyle='--', alpha=0.6)

ax0 = fig.add_subplot(gs[0, :])
bars(ax0, df1, 'Qualitative Dataset', (0.5, 1))

for i, (df, ylim, title) in enumerate([(df2, (0.75, 1), 'MS Dataset'),
                                       (df3, (0.4, 0.95), 'IC50<500 Dataset'),
                                       (df4, (0.4, 0.95), 'IC50<1000 Dataset')]):
    bars(fig.add_subplot(gs[1, i]), df, title, ylim)

# the legend sits clear of the tick labels: the two published tools carry version
# suffixes, so the bottom row's labels are long enough to reach it at -0.04
fig.legend([fx.METRIC_NAMES[m] for m in METRICS], loc='lower center',
           ncol=len(METRICS), bbox_to_anchor=(0.5, -0.09), frameon=False)
plt.savefig('fig2.pdf', bbox_inches='tight')
plt.savefig('fig2.svg', bbox_inches='tight')
plt.show()